# 16 · Spark ML + Structured Streaming — Scoring de Risco em Tempo Real

🎯 **Objetivo:** fechar o ciclo entre os dois últimos módulos do curso. O notebook 15
treinou, ajustou e **salvou** um `PipelineModel` capaz de prever o risco de uma avaliação
ser negativa. Este notebook faz a pergunta que fica pendente depois de qualquer treino em
lote: *e quando os dados chegam um a um, ao vivo?* Vamos recarregar aquele Pipeline e
aplicá-lo **dentro de um stream** — o mesmo padrão de simulação de arquivos dos
notebooks 12-14, agora com um modelo de Machine Learning no lugar de uma agregação.

Nenhum retreino acontece aqui. `PipelineModel.transform()` é uma função determinística
sobre um DataFrame — e um DataFrame de streaming é, para efeitos de `.transform()`, só
mais um DataFrame. É por isso que o mesmo Pipeline treinado em lote roda, sem nenhuma
alteração de código, sobre um `readStream`.

---
### 🔤 O que você vai praticar

1. **Carregar um `PipelineModel` persistido** — `PipelineModel.load(...)`, sem precisar
   reconstruir nenhum estágio manualmente
2. **Aplicar um modelo de ML dentro de `foreachBatch`** — a ponte entre um Pipeline
   treinado em lote e a inferência em tempo real
3. **Verificar a detecção ao vivo** — publicar avaliações da versão `4.9.1` (o bug que o
   notebook 15 descobriu) e observar o risco previsto disparar no mesmo micro-lote
4. **Uma fila de contato proativo** — persistir só as avaliações de alto risco em uma
   camada Gold, pronta para outro sistema consumir

## Setup: SparkSession, o Pipeline treinado e o schema do stream

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder.appName("app-01")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.executor.memory", "2g")
    .config("spark.executor.cores", "2")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

spark

In [ ]:
import pyspark.sql.functions as F
from pyspark.ml import PipelineModel
from pyspark.ml.functions import vector_to_array

CAMINHO_MODELO_RISCO = "../data/models/nb15_risco_avaliacao"

modelo_risco = PipelineModel.load(CAMINHO_MODELO_RISCO)
print(f"Pipeline carregado — {len(modelo_risco.stages)} estágios:")
for estagio in modelo_risco.stages:
    print(" ", type(estagio).__name__)

📌 **Se este load falhar:** rode o notebook 15 primeiro — ele treina e persiste este
mesmo Pipeline em `../data/models/nb15_risco_avaliacao`.

## Simulando o stream de avaliações

Mesma técnica dos notebooks 12-14: uma pasta `landing` recebe arquivos JSON Lines novos,
lidos como uma *Input Table* infinita via `readStream`. O schema replica os campos que o
Pipeline espera (`canal`, `versao_app`, `tempo_atendimento_min`) mais os campos que só
usamos para **conferir** a previsão (`nota`, `comentario`) — nunca como atributo.

In [ ]:
import json
import os
import shutil
import time
import uuid
from datetime import datetime, timedelta

from pyspark.sql.types import (
    DoubleType,
    IntegerType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)

BASE = "../data/streaming/nb16"
CAMINHO_GOLD_ALERTAS = "../data/gold/nb16_alertas_risco"
shutil.rmtree(BASE, ignore_errors=True)
shutil.rmtree(CAMINHO_GOLD_ALERTAS, ignore_errors=True)
os.makedirs(BASE, exist_ok=True)


def emitir_lote(eventos: list[dict], pasta: str = BASE) -> None:
    # Publica um lote de avaliações como um novo arquivo JSON Lines (rename atômico)
    os.makedirs(pasta, exist_ok=True)
    nome_arquivo = f"{uuid.uuid4().hex}.json"
    caminho_tmp = f"{pasta}/.{nome_arquivo}.tmp"
    caminho_final = f"{pasta}/{nome_arquivo}"
    with open(caminho_tmp, "w") as f:
        for evento in eventos:
            f.write(json.dumps(evento) + "\n")
    os.rename(caminho_tmp, caminho_final)
    print(f"📨 lote de {len(eventos)} avaliação(ões) publicado em landing/{nome_arquivo}")


def aguardar_processamento(query, timeout: int = 40) -> None:
    # Espera o próximo micro-batch com dados novos, em vez de adivinhar um sleep fixo
    batch_visto = query.lastProgress["batchId"] if query.lastProgress else -1
    inicio = time.time()
    while time.time() - inicio < timeout:
        time.sleep(1)
        progresso = query.lastProgress
        if progresso and progresso["batchId"] > batch_visto and progresso.get("numInputRows", 0) > 0:
            time.sleep(1)  # pequena folga para o foreachBatch concluir a escrita em disco
            return
    print(f"⚠️  tempo esgotado ({timeout}s) esperando o próximo micro-batch")


schema_avaliacao_stream = StructType([
    StructField("id_avaliacao", StringType()),
    StructField("id_empresa", IntegerType()),
    StructField("canal", StringType()),
    StructField("nota", IntegerType()),
    StructField("comentario", StringType()),
    StructField("versao_app", StringType()),           # null fora do canal App
    StructField("tempo_atendimento_min", DoubleType()),  # null fora do Call Center
    StructField("data_evento", TimestampType()),
])

## O pipeline de scoring: `foreachBatch` como ponte batch → streaming

`foreachBatch` entrega cada micro-lote como um DataFrame **estático** comum — é aqui que
aplicamos exatamente a mesma engenharia de atributos determinística do notebook 15
(`fillna` de `versao_app` e o indicador de ausência de `tempo_atendimento_min`) antes de
chamar `modelo_risco.transform()`. Avaliações sinalizadas como risco alto (`prediction == 1.0`)
são gravadas na camada Gold — uma fila de contato proativo para outro sistema consumir.

In [ ]:
def pontuar_e_publicar(lote_df, id_do_lote: int) -> None:
    if lote_df.rdd.isEmpty():
        return

    lote_preparado = (
        lote_df
        .fillna({"versao_app": "nao_app"})
        .withColumn("tempo_atendimento_ausente", F.col("tempo_atendimento_min").isNull().cast("double"))
    )
    pontuado = modelo_risco.transform(lote_preparado).withColumn(
        "risco_previsto", F.round(vector_to_array("probability")[1], 3)
    )

    resultado = pontuado.select(
        "id_avaliacao", "canal", "versao_app", "tempo_atendimento_min",
        "nota", "risco_previsto", "prediction",
    ).orderBy(F.col("risco_previsto").desc())

    print(f"\n=== micro-lote {id_do_lote} — {resultado.count()} avaliação(ões) pontuada(s) ===")
    resultado.show(20, truncate=False)

    alertas = pontuado.filter(F.col("prediction") == 1.0)
    n_alertas = alertas.count()
    if n_alertas > 0:
        alertas.select(
            "id_avaliacao", "id_empresa", "canal", "versao_app", "risco_previsto", "nota"
        ).write.mode("append").parquet(CAMINHO_GOLD_ALERTAS)
        print(f"🚨 {n_alertas} avaliação(ões) sinalizada(s) para contato proativo → {CAMINHO_GOLD_ALERTAS}")
    else:
        print("✅ nenhuma avaliação de alto risco neste lote")


avaliacoes_stream = spark.readStream.format("json").schema(schema_avaliacao_stream).load(BASE)
print(f"avaliacoes_stream.isStreaming = {avaliacoes_stream.isStreaming}")

query_scoring = (
    avaliacoes_stream.writeStream
    .foreachBatch(pontuar_e_publicar)
    .trigger(processingTime="2 seconds")
    .start()
)

## Cenário 1: tráfego normal — a maioria deve receber risco baixo

In [ ]:
BASE_TIME = datetime(2026, 7, 30, 9, 0, 0)

lote_normal = [
    {"id_avaliacao": "R1001", "id_empresa": 9, "canal": "App", "nota": 5,
     "comentario": "Ótimo atendimento, super recomendo", "versao_app": "5.0.0",
     "tempo_atendimento_min": None, "data_evento": BASE_TIME.isoformat()},
    {"id_avaliacao": "R1002", "id_empresa": 16, "canal": "App", "nota": 2,
     "comentario": "Produto chegou com pequenos problemas", "versao_app": "4.8.0",
     "tempo_atendimento_min": None, "data_evento": (BASE_TIME + timedelta(seconds=5)).isoformat()},
    {"id_avaliacao": "R1003", "id_empresa": 33, "canal": "Site", "nota": 4,
     "comentario": "Boa experiência no geral", "versao_app": None,
     "tempo_atendimento_min": None, "data_evento": (BASE_TIME + timedelta(seconds=10)).isoformat()},
    {"id_avaliacao": "R1004", "id_empresa": 44, "canal": "Call Center", "nota": 5,
     "comentario": "Atendimento rápido e resolveu na hora", "versao_app": None,
     "tempo_atendimento_min": 6.5, "data_evento": (BASE_TIME + timedelta(seconds=15)).isoformat()},
    {"id_avaliacao": "R1005", "id_empresa": 12, "canal": "Call Center", "nota": 1,
     "comentario": "Atendimento muito ruim", "versao_app": None,
     "tempo_atendimento_min": 28.0, "data_evento": (BASE_TIME + timedelta(seconds=20)).isoformat()},
]

emitir_lote(lote_normal)
aguardar_processamento(query_scoring)

📌 **Leitura do lote:** `R1005` (Call Center, 28 minutos de espera) já deveria puxar o
risco para cima — é exatamente o sinal que o notebook 15 mediu com correlação de ~0.6
entre `tempo_atendimento_min` e nota ruim. As demais ficam perto da taxa-base (~15-19%).

## Cenário 2: o bug da versão `4.9.1` chega ao vivo

O mesmo padrão que o notebook 09 só conseguiu **descrever** depois do fato e que o
notebook 15 aprendeu a **prever** em lote — agora chegando em tempo real.

In [ ]:
lote_bug = [
    {"id_avaliacao": f"R200{i}", "id_empresa": 5 + i, "canal": "App", "nota": 1,
     "comentario": "Produto chegou com defeito", "versao_app": "4.9.1",
     "tempo_atendimento_min": None, "data_evento": (BASE_TIME + timedelta(minutes=1, seconds=i)).isoformat()}
    for i in range(5)
]

emitir_lote(lote_bug)
aguardar_processamento(query_scoring)

📌 **Se o `risco_previsto` desta rodada saiu perto de 0.9+ para todas as 5 avaliações**,
o Pipeline treinado em lote no notebook 15 acabou de sinalizar, em tempo real e sem
nenhum código específico para "versão 4.9.1", o mesmíssimo bug que motivou aquele
notebook inteiro. Isso é o que "colocar um modelo em produção" quer dizer na prática.

## Cenário 3: depois do hotfix — a versão nova volta a marcar risco baixo

Fechando o arco: a equipe de produto corrige o bug e lança a versão `5.0.1`.

In [ ]:
lote_pos_fix = [
    {"id_avaliacao": f"R300{i}", "id_empresa": 20 + i, "canal": "App", "nota": 5,
     "comentario": "Superou minhas expectativas", "versao_app": "5.0.1",
     "tempo_atendimento_min": None, "data_evento": (BASE_TIME + timedelta(minutes=2, seconds=i)).isoformat()}
    for i in range(4)
]

emitir_lote(lote_pos_fix)
aguardar_processamento(query_scoring)

📌 **Atenção ao detalhe:** `5.0.1` nunca apareceu no treino do notebook 15 — o
`StringIndexer(handleInvalid="keep")` é exatamente o que evita que este Pipeline
**quebre em produção** diante de uma categoria nova, agrupando-a num índice extra em vez
de lançar exceção. É a mesma armadilha do Museu dos Erros do notebook 15, só que agora
resolvida por uma escolha de parâmetro feita ali, e paga aqui.

## Conferindo a fila de contato proativo (camada Gold)

In [ ]:
alertas_gold = spark.read.parquet(CAMINHO_GOLD_ALERTAS)
print(f"Total de avaliações sinalizadas para contato proativo: {alertas_gold.count()}")
alertas_gold.orderBy(F.col("risco_previsto").desc()).show(truncate=False)

📌 Note que **nenhuma avaliação da versão `5.0.1`** entrou nesta fila — o hotfix "convenceu"
o modelo. Esta tabela Parquet é a interface de saída: qualquer outro sistema (um CRM, uma
fila de discagem, um dashboard de atendimento) pode consumi-la sem saber nada sobre Spark
ou MLlib.

### Uma nota sobre deriva (*drift*) — o que este notebook não cobre

Este Pipeline foi treinado uma vez, no notebook 15, sobre uma fotografia dos dados. Em
produção de verdade, o comportamento dos clientes muda — um novo canal aparece, um novo
concorrente rouba justamente os clientes mais satisfeitos, a distribuição de `versao_app`
migra por completo em poucas semanas. Nada disso derruba o job (como acabamos de ver com
a versão `5.0.1`), mas o modelo pode ir ficando **estatisticamente desatualizado** sem
avisar. A resposta de produção é monitorar a distribuição das previsões ao longo do tempo
e reexecutar o notebook 15 periodicamente — não é um problema de código, é um problema de
processo.

In [ ]:
query_scoring.stop()
spark.stop()

---
🎉 **Parabéns!** Você completou o notebook 16 — e o laboratório de Spark ML.

Você aprendeu:
- Que um `PipelineModel` treinado em lote roda **sem nenhuma alteração de código** sobre
  um DataFrame de streaming — `.transform()` é só uma função determinística
- Como usar `foreachBatch` como a ponte entre a engenharia de atributos determinística
  (o `fillna`/indicador de nulo do notebook 15) e a inferência de um modelo já treinado
- Como observar, ao vivo, um modelo sinalizando um padrão real de negócio (o bug da
  versão `4.9.1`) no exato micro-lote em que ele chega
- Por que `StringIndexer(handleInvalid="keep")` é o que evita que uma categoria nova em
  produção (`5.0.1`) derrube o job inteiro
- Como persistir só as previsões de alto risco em uma camada Gold — uma fila de contato
  pronta para outro sistema consumir, sem nenhum acoplamento a Spark ou MLlib

### 📝 Exercícios propostos

1. Publique um lote de Call Center com `tempo_atendimento_min` crescente (5, 15, 25, 35
   minutos) e trace como `risco_previsto` reage — existe um "ponto de virada" visível?
2. Troque `handleInvalid="keep"` por `"error"` no `StringIndexer` do notebook 15, retreine,
   e rode o Cenário 3 de novo — confirme, ao vivo, o crash que o Museu dos Erros descreveu.
3. Adapte `pontuar_e_publicar` para também gravar, num segundo caminho Parquet, as
   avaliações de risco **baixo** — e meça quanto espaço em disco cada fila ocupa por hora
   de tráfego simulado.

---